# DEVKIT AI — Huihui-Qwen3.8-27B-abliterated (Q4_K_M, No-Think, Tên Miền Cố Định)
## Kaggle (GPU T4 x2) + Ollama + No-Think Proxy + Ngrok Static Domain

Notebook này giúp bạn chạy **`hf.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF:Q4_K_M`** (~16.6 GB) với **1 đường link cố định vĩnh viễn** (không phải copy dán lại URL vào VS Code / Cline mỗi lần mở máy):
- **Mặc định TẮT Thinking** (chỉ bật khi bạn gõ `/think` trong tin nhắn).
- **URL cố định miễn phí** qua **Ngrok Static Domain** (`https://<ten-cua-ban>.ngrok-free.dev/v1`).

**Cài đặt bắt buộc trên Kaggle (menu `Settings` góc trên bên trái):**
1. **Accelerator:** chọn **GPU T4 x2** (32 GB VRAM).
2. **Turn on internet:** bật **On**.

## 1. Kiểm tra GPU và dung lượng ổ đĩa

In [6]:
!nvidia-smi
!df -h / /kaggle/working 2>/dev/null || df -h /

Sat Nov 14 15:48:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Cài đặt Ollama và Ngrok
Tự động xử lý quyền `root` trên Kaggle để không bị lỗi `sudo`.

In [7]:
import os, shutil, subprocess

def run_cmd(cmd):
    print(">>", cmd)
    subprocess.run(cmd, shell=True, check=True)

prefix = "" if os.geteuid() == 0 else "sudo "

if not shutil.which("zstd") or not shutil.which("curl"):
    run_cmd(f"{prefix}apt-get update -y -qq && {prefix}apt-get install -y -qq zstd curl tar")

if not shutil.which("ollama"):
    run_cmd("curl -fsSL https://ollama.com/install.sh | sh")

if not shutil.which("ngrok"):
    run_cmd("curl -fsSL https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz -o /tmp/ngrok.tgz")
    run_cmd(f"{prefix}tar -xzf /tmp/ngrok.tgz -C /usr/local/bin")
    run_cmd(f"{prefix}chmod +x /usr/local/bin/ngrok")
    run_cmd("rm -f /tmp/ngrok.tgz")

run_cmd("ollama --version")
run_cmd("ngrok version")

>> ollama --version
ollama version is 0.40.16
>> ngrok version
ngrok version 3.39.13


## 3. Khởi động Ollama server (Tối ưu VRAM cho 2× T4)

In [20]:
import os, subprocess, time, requests

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"

ollama_log_path = os.path.join(WORK_DIR, "ollama.log")
ollama_log = open(ollama_log_path, "w")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    env=os.environ.copy(),
    stdout=ollama_log,
    stderr=subprocess.STDOUT
)

for _ in range(30):
    try:
        r = requests.get("http://127.0.0.1:11434", timeout=2)
        if r.status_code == 200:
            print("Ollama server đã sẵn sàng:", r.text)
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError(f"Ollama không khởi động được. Xem log tại: {ollama_log_path}")

Ollama server đã sẵn sàng: Ollama is running


## 4. Tải model `hf.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF:Q4_K_M` (~16.6 GB)

In [ ]:
import subprocess, time

BASE_MODEL = "hf.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF:Q4_K_M"
print("Đang tải model 16.6 GB (vui lòng đợi 2-5 phút, không spam log)...")

with open("/tmp/pull.log", "w") as log_f:
    proc = subprocess.Popen(["ollama", "pull", BASE_MODEL], stdout=log_f, stderr=subprocess.STDOUT)
    start = time.time()
    while proc.poll() is None:
        time.sleep(15)
        print(f"  ... vẫn đang tải ({int(time.time() - start)}s)")

if proc.returncode != 0:
    print(open("/tmp/pull.log").read()[-2000:])
    raise RuntimeError("Tải model thất bại!")

# Tạo alias có sẵn num_ctx = 16384 cho AI Agent (không tốn thêm dung lượng)
modelfile = f"""FROM {BASE_MODEL}
PARAMETER num_ctx 16384
"""
open("/tmp/Modelfile", "w").write(modelfile)
subprocess.run(["ollama", "create", "qwen38-27b-agent", "-f", "/tmp/Modelfile"], check=True)

MODEL = "qwen38-27b-agent"
print("Đã sẵn sàng model:", MODEL)
!ollama list

]11;?\pulling manifest 
pulling 0c7cfe306049: 100% ▕██████████████████▏  16 GB                         
pulling c9a090646836: 100% ▕██████████████████▏ 931 MB                         
pulling 50153b537730: 100% ▕██████████████████▏  481 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 
]11;?\NAME                                                          ID              SIZE     MODIFIED      
hf.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF:Q4_K_M    c7dba207a3bf    17 GB    5 seconds ago    


## 5. Khởi động Smart No-Think Proxy (Cổng `11435` → `11434`)
Mặc định tắt Thinking (`reasoning_effort = "none"` / `think = false`). Gõ `/think` trong tin nhắn nếu muốn bật lại cho lượt đó.

In [ ]:
import json, threading, urllib.request, urllib.error
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

OLLAMA_UPSTREAM = "http://127.0.0.1:11434"
PROXY_PORT = 11435

def apply_nothink_rules(path, body_bytes):
    if not body_bytes or not any(p in path for p in ("/v1/chat/completions", "/api/chat", "/api/generate")):
        return body_bytes
    try:
        data = json.loads(body_bytes.decode("utf-8"))
    except Exception:
        return body_bytes

    want_think = False
    if isinstance(data.get("messages"), list):
        for msg in data["messages"]:
            content = msg.get("content")
            if isinstance(content, str):
                if "/think" in content:
                    want_think = True
                    msg["content"] = content.replace("/think", "").strip()
                if "/no_think" in content:
                    want_think = False
                    msg["content"] = content.replace("/no_think", "").strip()
                if "/nothink" in content:
                    want_think = False
                    msg["content"] = content.replace("/nothink", "").strip()
    elif isinstance(data.get("prompt"), str):
        if "/think" in data["prompt"]:
            want_think = True
            data["prompt"] = data["prompt"].replace("/think", "").strip()
        if "/no_think" in data["prompt"]:
            want_think = False
            data["prompt"] = data["prompt"].replace("/no_think", "").strip()

    if "/v1/" in path:
        data["reasoning_effort"] = "medium" if want_think else "none"
    else:
        data["think"] = True if want_think else False

    return json.dumps(data, ensure_ascii=False).encode("utf-8")

class OllamaNoThinkProxy(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass

    def _forward(self):
        length = int(self.headers.get("Content-Length", 0))
        raw_body = self.rfile.read(length) if length > 0 else b""
        if self.command == "POST":
            raw_body = apply_nothink_rules(self.path, raw_body)

        headers = {k: v for k, v in self.headers.items() if k.lower() not in ("host", "content-length")}
        if raw_body:
            headers["Content-Length"] = str(len(raw_body))

        req = urllib.request.Request(
            OLLAMA_UPSTREAM + self.path,
            data=raw_body if self.command in ("POST", "PUT", "PATCH") else None,
            headers=headers,
            method=self.command,
        )
        try:
            with urllib.request.urlopen(req, timeout=600) as resp:
                self.send_response(resp.status)
                for k, v in resp.getheaders():
                    if k.lower() not in ("transfer-encoding", "connection"):
                        self.send_header(k, v)
                self.end_headers()
                while True:
                    chunk = resp.read(4096)
                    if not chunk:
                        break
                    self.wfile.write(chunk)
                    self.wfile.flush()
        except urllib.error.HTTPError as e:
            err_body = e.read()
            self.send_response(e.code)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(err_body)))
            self.end_headers()
            self.wfile.write(err_body)
        except Exception as e:
            msg = json.dumps({"error": str(e)}).encode("utf-8")
            self.send_response(502)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(msg)))
            self.end_headers()
            self.wfile.write(msg)

    def do_GET(self): self._forward()
    def do_POST(self): self._forward()
    def do_PUT(self): self._forward()
    def do_DELETE(self): self._forward()
    def do_OPTIONS(self): self._forward()

if "proxy_server" in globals() and proxy_server is not None:
    try:
        proxy_server.shutdown()
        proxy_server.server_close()
    except Exception:
        pass

import socket

class ReusableHTTPServer(ThreadingHTTPServer):
    allow_reuse_address = True

try:
    with socket.create_connection(("127.0.0.1", PROXY_PORT), timeout=1):
        print(f"Smart No-Think Proxy đã đang chạy sẵn tại http://127.0.0.1:{PROXY_PORT}")
except OSError:
    proxy_server = ReusableHTTPServer(("127.0.0.1", PROXY_PORT), OllamaNoThinkProxy)
    proxy_thread = threading.Thread(target=proxy_server.serve_forever, daemon=True)
    proxy_thread.start()
    print(f"Smart No-Think Proxy đang chạy tại http://127.0.0.1:{PROXY_PORT}")

Smart No-Think Proxy đã đang chạy sẵn tại http://127.0.0.1:11435


## 6. Kiểm tra nhanh model qua Proxy (Mặc định tắt Thinking)

In [ ]:
# import requests

# resp = requests.post(
#     "http://127.0.0.1:11435/v1/chat/completions",
#     json={
#         "model": MODEL,
#         "messages": [{"role": "user", "content": "Trả lời ngắn gọn bằng tiếng Việt trong 1 câu: 2 + 2 bằng mấy?"}],
#         "stream": False
#     },
#     timeout=300
# )
# print("HTTP:", resp.status_code)
# resp.raise_for_status()
# msg = resp.json()["choices"][0]["message"]
# print("Trả lời:", msg.get("content", ""))
# print("Reasoning (rỗng là thành công):", repr(msg.get("reasoning", "")))

HTTP: 200
Trả lời: 2 + 2 bằng 4.
Reasoning (rỗng là thành công): ''


## 7. Mở Ngrok với Tên Miền Cố Định (Không đổi URL mỗi lần chạy)

**Cách lấy 2 thông tin bên dưới (chỉ làm 1 lần):**
1. Đăng ký/đăng nhập miễn phí tại [dashboard.ngrok.com](https://dashboard.ngrok.com).
2. Vào **Your Authtoken** → copy token dán vào `NGROK_AUTHTOKEN`.
3. Vào **Domains** (hoặc **Universal Gateway → Domains**) → bấm tạo/xem tên miền tĩnh miễn phí (dạng `xxxx.ngrok-free.dev`) dán vào `NGROK_DOMAIN` (không cần ghi `https://`).
*(Mẹo bảo mật: Nếu muốn chia sẻ notebook, hãy lưu 2 giá trị này trong **Add-ons → Secrets** của Kaggle với tên `NGROK_AUTHTOKEN` và `NGROK_DOMAIN` thay vì dán trực tiếp vào code).*

In [ ]:
import os, subprocess, time, requests

# Cách 1: Tự điền trực tiếp vào đây (nếu notebook để Private)
NGROK_AUTHTOKEN = ""
NGROK_DOMAIN = ""  # Ví dụ: "my-ai-server.ngrok-free.dev"

# Cách 2: Tự động đọc từ Kaggle Secrets nếu bạn đã lưu ở Add-ons -> Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    if not NGROK_AUTHTOKEN:
        NGROK_AUTHTOKEN = secrets.get_secret("NGROK_AUTHTOKEN")
    if not NGROK_DOMAIN:
        NGROK_DOMAIN = secrets.get_secret("NGROK_DOMAIN")
except Exception:
    pass

NGROK_DOMAIN = NGROK_DOMAIN.replace("https://", "").replace("http://", "").strip("/")
if not NGROK_AUTHTOKEN or not NGROK_DOMAIN:
    raise ValueError("Bạn chưa điền NGROK_AUTHTOKEN hoặc NGROK_DOMAIN ở Bước 7!")

subprocess.run(["ngrok", "config", "add-authtoken", NGROK_AUTHTOKEN], check=True)

if "ngrok_process" in globals() and ngrok_process is not None:
    try:
        ngrok_process.terminate()
    except Exception:
        pass

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
ngrok_log = open(os.path.join(WORK_DIR, "ngrok.log"), "w")
ngrok_process = subprocess.Popen(
    ["ngrok", "http", f"--url={NGROK_DOMAIN}", "11435"],
    stdout=ngrok_log,
    stderr=subprocess.STDOUT
)

PUBLIC_URL = f"https://{NGROK_DOMAIN}"
for _ in range(20):
    time.sleep(1.5)
    try:
        r = requests.get(PUBLIC_URL + "/api/tags", timeout=5)
        if r.status_code == 200:
            break
    except Exception:
        pass

print("PUBLIC URL CỐ ĐỊNH:", PUBLIC_URL)
print("Base URL (OpenAI Compatible):", PUBLIC_URL + "/v1")
print("Model ID:", MODEL)

Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml
PUBLIC URL CỐ ĐỊNH: https://fedora-muzzle-neatly.ngrok-free.dev
Base URL (OpenAI Compatible): https://fedora-muzzle-neatly.ngrok-free.dev/v1
Model ID: hf.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF:Q4_K_M


## 8. Kiểm tra kết nối từ bên ngoài qua Tên miền cố định

In [ ]:
# import requests

# r = requests.post(
#     PUBLIC_URL + "/v1/chat/completions",
#     json={
#         "model": MODEL,
#         "messages": [{"role": "user", "content": "Reply exactly: OK"}],
#         "stream": False
#     },
#     headers={"Authorization": "Bearer ollama"},
#     timeout=300
# )
# print("HTTP:", r.status_code)
# print(r.text[:1000])
# r.raise_for_status()

HTTP: 200
{"id":"chatcmpl-810","object":"chat.completion","created":1794671879,"model":"hf.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF:Q4_K_M","system_fingerprint":"fp_ollama","choices":[{"index":0,"message":{"role":"assistant","content":"OK"},"finish_reason":"stop"}],"usage":{"prompt_tokens":16,"prompt_tokens_details":{"cached_tokens":0},"completion_tokens":2,"total_tokens":18}}



## 9. Cấu hình 1 lần duy nhất vào VS Code / Cline / OpenCode

| Trường | Giá trị |
| :--- | :--- |
| **Provider** | OpenAI Compatible |
| **Base URL** | `https://<NGROK_DOMAIN>/v1` (cố định vĩnh viễn) |
| **API Key** | `ollama` |
| **Model ID** | `hf.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF:Q4_K_M` |

**Mẹo dùng:** Bình thường model tự tắt Thinking để trả lời cực nhanh. Khi cần suy nghĩ sâu, chỉ cần thêm `/think` vào tin nhắn.